In [1]:
import numpy as np
import pandas as pd
import perming

In [2]:
WT_SEQ = (
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPW"
    "PTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKF"
    "EGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNI"
    "EDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAG"
    "ITHGMDELYK"
)
assert len(WT_SEQ) == 238

In [3]:

AA_LIST = np.array(list("ACDEFGHIKLMNPQRSTVWY"))
AA2IDX = {aa: i for i, aa in enumerate(AA_LIST)}

L = len(WT_SEQ)          # 238
D_ONEHOT = L * 20        # 4760
D_COUNT = L + 20 + 1     # 259

In [4]:
def _is_empty(mut_str):
    if mut_str is None:
        return True
    if isinstance(mut_str, float) and pd.isna(mut_str):
        return True
    return str(mut_str).strip() == ""


def parse_mutations(mut_str):
    if _is_empty(mut_str):
        return []
    out = []
    for tok in str(mut_str).strip().split(":"):
        tok = tok.strip()
        if len(tok) < 3:
            continue
        mut_aa = tok[-1]
        if mut_aa == "*":
            continue
        try:
            pos = int(tok[1:-1]) - 1  # 1-indexed -> 0-indexed
        except ValueError:
            continue
        if 0 <= pos < L:
            out.append((pos, mut_aa))
    return out

def encode_onehot(mut_str):
    x = np.zeros((L, 20), dtype=np.float32)
    for i, aa in enumerate(WT_SEQ):
        x[i, AA2IDX[aa]] = 1.0

    for pos, mut_aa in parse_mutations(mut_str):
        x[pos, :] = 0
        x[pos, AA2IDX[mut_aa]] = 1.0

    return x.flatten()

def encode_count(mut_str):
    pos_mut = np.zeros(L, dtype=np.float32)
    aa_mut = np.zeros(20, dtype=np.float32)
    n_mut = 0

    for pos, mut_aa in parse_mutations(mut_str):
        pos_mut[pos] = 1
        aa_mut[AA2IDX[mut_aa]] += 1
        n_mut += 1

    return np.concatenate([pos_mut, aa_mut, [n_mut]])

def build_dataset(tsv_path, check_wt=True):
    df = pd.read_csv(tsv_path, sep="\t")
    N = len(df)

    X_onehot = np.zeros((N, D_ONEHOT), dtype=np.float32)
    X_count = np.zeros((N, D_COUNT), dtype=np.float32)
    y = np.zeros(N, dtype=np.float32)
    mask_valid = np.zeros(N, dtype=bool)
    mutation_count = np.zeros(N, dtype=np.int32)
    meta = np.empty(N, dtype=object)

    raw_col = df["aaMutations"]

    if check_wt:
        tot = ok = 0
        for s in raw_col:
            for tok in ([] if _is_empty(s) else str(s).strip().split(":")):
                tok = tok.strip()
                if len(tok) < 3:
                    continue
                try:
                    pos = int(tok[1:-1]) - 1
                except ValueError:
                    continue
                if 0 <= pos < L:
                    tot += 1
                    ok += (WT_SEQ[pos] == tok[0])
        if tot:
            print(f"[check] WT Residue matching rate: {ok}/{tot} = {ok / tot:.4f}")
            if ok / tot < 0.9:
                print("The matching rate is too low, and WT_SQ may be misaligned with the mutation number")

    for i, row in enumerate(df.itertuples()):
        X_onehot[i] = encode_onehot(row.aaMutations)
        X_count[i] = encode_count(row.aaMutations)
        mutation_count[i] = len(parse_mutations(row.aaMutations))
        meta[i] = "" if _is_empty(row.aaMutations) else str(row.aaMutations).strip()

        if not pd.isna(row.medianBrightness):
            y[i] = row.medianBrightness
            mask_valid[i] = True

    print(f"Samples : {N}")
    print(f"valid y : {mask_valid.sum()}")
    print(f"X_onehot: {X_onehot.shape}")
    print(f"X_count : {X_count.shape}")
    return dict(
        X_onehot=X_onehot,
        X_count=X_count,
        y=y,
        mask_valid=mask_valid,
        mutation_count=mutation_count,
        wt_seq=np.array(list(WT_SEQ)),
        aa_list=AA_LIST,
        meta=meta,
    )

In [ ]:

data = build_dataset(
    tsv_path="../data/amino_acid_genotypes_to_brightness.tsv",
)


Samples : 54025
valid y : 54025
X_onehot: (54025, 4760)
X_count : (54025, 259)


In [6]:
X = data["X_count"][data["mask_valid"]]
y = data["y"][data["mask_valid"]]

In [7]:
main = perming.Box(259, 1, (100,), criterion='MSELoss', batch_size=128, activation='relu', inplace_on=True, solver='adam', learning_rate_init=0.01)
main.print_config()
main.data_loader(X, y, random_seed=0)
main.train_val(num_epochs=2, interval=100, early_stop=True)
main.test()

MLP(
  (mlp): Sequential(
    (Linear0): Linear(in_features=259, out_features=100, bias=True)
    (Activation0): ReLU(inplace=True)
    (Linear1): Linear(in_features=100, out_features=1, bias=True)
  )
)
Process stop at epoch [1/2] with patience 10 within tolerance 0.001
loss of Box on the 5504 test dataset: 1.1210064888000488.


OrderedDict([('problem', 'regression'),
             ('loss',
              {'train': 1.1532363891601562,
               'val': 0.15943817794322968,
               'test': 1.1210064888000488})])